# Imbalanced Classification Evaluation

This notebook generates a 95:5 synthetic dataset, trains a baseline logistic-regression classifier, implements custom binary metrics, plots ROC and precision-recall curves, studies thresholds, and compares F1 averaging strategies.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    classification_report)

X, y = make_classification(n_samples=8000, n_features=20, n_informative=6,
    n_redundant=4, weights=[0.95, 0.05], class_sep=1.0, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
print('Train positive rate:', round(y_train.mean(), 4))
print('Test positive rate:', round(y_test.mean(), 4))

## Custom metric calculations

In [ ]:
def custom_binary_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
            'precision': precision, 'recall': recall, 'f1': f1}

model = LogisticRegression(max_iter=2000, random_state=42)
model.fit(X_train, y_train)
prob = model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.50).astype(int)
print(custom_binary_metrics(y_test, pred))
print('Accuracy:', round(accuracy_score(y_test, pred), 4))
print('ROC-AUC:', round(roc_auc_score(y_test, prob), 4))
print('Average precision:', round(average_precision_score(y_test, prob), 4))
print(classification_report(y_test, pred, digits=4, zero_division=0))

## ROC and precision-recall curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, prob)
plt.figure(figsize=(7, 5)); plt.plot(fpr, tpr, label=f'AUC={roc_auc_score(y_test, prob):.3f}')
plt.plot([0, 1], [0, 1], '--'); plt.xlabel('False positive rate'); plt.ylabel('Recall')
plt.title('ROC curve'); plt.legend(); plt.show()

p, r, _ = precision_recall_curve(y_test, prob)
plt.figure(figsize=(7, 5)); plt.plot(r, p, label=f'AP={average_precision_score(y_test, prob):.3f}')
plt.axhline(y.mean(), linestyle='--', label=f'Prevalence={y.mean():.3f}')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-recall curve'); plt.legend(); plt.show()

## Threshold trade-offs

In [ ]:
rows = []
for threshold in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    pred_t = (prob >= threshold).astype(int)
    m = custom_binary_metrics(y_test, pred_t)
    rows.append({'threshold': threshold, **m})
threshold_results = pd.DataFrame(rows)
display(threshold_results.round(3))
threshold_results.plot(x='threshold', y=['precision', 'recall', 'f1'], marker='o', ylim=(0, 1), figsize=(7, 5))
plt.title('Threshold trade-offs'); plt.ylabel('Score'); plt.show()

## Macro, micro, and weighted F1

In [ ]:
f1_summary = pd.DataFrame({
    'averaging': ['macro', 'micro', 'weighted'],
    'F1': [f1_score(y_test, pred, average='macro', zero_division=0),
           f1_score(y_test, pred, average='micro', zero_division=0),
           f1_score(y_test, pred, average='weighted', zero_division=0)]
})
display(f1_summary.round(4))
print('In single-label classification, micro F1 equals accuracy; weighted F1 is influenced by the majority class.')

## Conclusion

For rare fraud, do not rely on accuracy alone. Use precision and recall to describe the operating point, the precision-recall curve / average precision to assess positive-class retrieval, and ROC-AUC as a complementary ranking metric. Select a threshold using validation data and business costs.